In [1]:
# ============================================================
# DAY 4 — FUNDAMENTAL SCORING SYSTEM
# AI Stock Screener — Indian Markets
# CLEAN FINAL VERSION
# ============================================================
 
# ── CELL 1: IMPORTS + LOAD DATA ─────────────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')
 
# Load metrics and survival filter results
metrics_df       = pd.read_csv('../data/fundamental_metrics.csv')
survival_results = pd.read_csv('../data/survival_filter_results.csv')
 
# Load raw data (needed for financial company scoring)
with open('../data/raw_stock_data.pkl', 'rb') as f:
    all_stock_data = pickle.load(f)
 
# Get only stocks that passed survival filter
passed_symbols = survival_results[
    survival_results['Passed Survival'] == True
]['Symbol'].tolist()
 
passed_df = metrics_df[
    metrics_df['Symbol'].isin(passed_symbols)
].copy().reset_index(drop=True)
 
print(f"Total stocks in metrics : {len(metrics_df)}")
print(f"Passed survival filter  : {len(passed_df)}")
print(f"\nStocks to score:")
print(passed_df['Symbol'].tolist())

Total stocks in metrics : 45
Passed survival filter  : 37

Stocks to score:
['TCS', 'INFY', 'HDFCBANK', 'RELIANCE', 'WIPRO', 'PERSISTENT', 'LTTS', 'COFORGE', 'HAPPSTMNDS', 'CHOLAFIN', 'MUTHOOTFIN', 'MANAPPURAM', 'CREDITACC', 'GARFIBRES', 'SUPRAJIT', 'HIMATSEIDE', 'HAL', 'BEL', 'MIDHANI', 'PARAS', 'VINATIORGA', 'FINEORG', 'GALAXYSURF', 'SUNPHARMA', 'DIVISLAB', 'APLLTD', 'GRANULES', 'IPCALAB', 'TITAN', 'HAVELLS', 'VGUARD', 'SYMPHONY', 'WONDERLA', 'PIIND', 'RALLIS', 'DHANUKA', 'ABSLAMC']


In [2]:
# ── CELL 2: SCORING HELPER FUNCTIONS ────────────────────────
def score_metric_level(value, thresholds, reverse=False):
    """
    Score a metric based on absolute level
    thresholds = [excellent, good, acceptable]
    reverse = True means lower is better (e.g. Debt/Equity)
    Returns 0-3
    """
    if pd.isna(value):
        return None
    if reverse:
        if value <= thresholds[0]:   return 3
        elif value <= thresholds[1]: return 2
        elif value <= thresholds[2]: return 1
        else:                        return 0
    else:
        if value >= thresholds[0]:   return 3
        elif value >= thresholds[1]: return 2
        elif value >= thresholds[2]: return 1
        else:                        return 0
 
 
def score_metric_trend(current, previous, higher_is_better=True):
    """
    Score trend direction
    Returns 0-2
    """
    if pd.isna(current) or pd.isna(previous):
        return None
    if higher_is_better:
        if current > previous * 1.05:    return 2  # improved >5%
        elif current >= previous * 0.95: return 1  # stable
        else:                            return 0  # deteriorated
    else:
        if current < previous * 0.95:    return 2  # improved >5%
        elif current <= previous * 1.05: return 1  # stable
        else:                            return 0  # deteriorated
 
 
def safe_score(level_score, trend_score):
    """Combine level and trend scores safely, handles None"""
    if level_score is None and trend_score is None:
        return None
    if level_score is None:
        return trend_score
    if trend_score is None:
        return level_score
    return level_score + trend_score
 
 
print("Scoring helper functions loaded ✅")

Scoring helper functions loaded ✅


In [3]:
# ── CELL 3: HISTORICAL SCORE — STANDARD COMPANIES (40 pts) ──
# Revenue growth  : 15 pts (5Y CAGR → 10Y → Max as fallback)
# Profit growth   : 15 pts (5Y CAGR → 10Y → Max as fallback)
# Operating margin: 5 pts  (5Y average)
# ROE             : 5 pts
# Maturity discount: <5Y = 50%, 5-8Y = 20%, 8Y+ = 0%
 
def calculate_historical_score(row):
    """
    Historical performance score for standard companies (40 pts)
    Uses 5Y CAGR as primary, falls back to 10Y then Max CAGR
    Applies maturity discount for short track record
    """
    score   = 0
    details = {}
 
    # ── DATA MATURITY CHECK ──────────────────────────────────
    # Discount scores for companies with short history
    # Prevents overfitting to a few good years
    cagr_years = float(
        row.get('Revenue CAGR Years') or
        row.get('Profit CAGR Years')  or 0
    )
    if cagr_years < 5:
        discount = 0.50
        maturity = 'Very Short (<5Y)'
    elif cagr_years < 8:
        discount = 0.20
        maturity = 'Short (5-8Y)'
    else:
        discount = 0.00
        maturity = 'Full (8Y+)'
 
    details['Data Years'] = cagr_years
    details['Maturity']   = maturity
    details['Discount']   = discount
 
    # ── REVENUE GROWTH (15 pts) ──────────────────────────────
    rev_growth = float(
        row.get('Revenue CAGR 5Y')  or
        row.get('Revenue CAGR 10Y') or
        row.get('Revenue CAGR Max') or 0
    )
    if rev_growth >= 20:   s = 15
    elif rev_growth >= 15: s = 12
    elif rev_growth >= 10: s = 9
    elif rev_growth >= 5:  s = 5
    else:                  s = 0
    score += s
    details['Revenue Growth Score'] = s
    details['Revenue CAGR Used']    = round(rev_growth, 1)
 
    # ── PROFIT GROWTH (15 pts) ───────────────────────────────
    profit_growth = float(
        row.get('Profit CAGR 5Y')  or
        row.get('Profit CAGR 10Y') or
        row.get('Profit CAGR Max') or 0
    )
    if profit_growth >= 20:  s = 15
    elif profit_growth >= 15:s = 12
    elif profit_growth >= 10:s = 9
    elif profit_growth >= 5: s = 5
    else:                    s = 0
    score += s
    details['Profit Growth Score'] = s
    details['Profit CAGR Used']    = round(profit_growth, 1)
 
    # ── OPERATING MARGIN (5 pts) ─────────────────────────────
    opm = float(
        row.get('Avg OPM 5Y') or
        row.get('Avg OPM 10Y') or 0
    )
    if opm >= 25:   s = 5
    elif opm >= 20: s = 4
    elif opm >= 15: s = 3
    elif opm >= 10: s = 2
    elif opm >= 5:  s = 1
    else:           s = 0
    score += s
    details['OPM Score']  = s
    details['Avg OPM 5Y'] = round(opm, 1)
 
    # ── ROE (5 pts) ──────────────────────────────────────────
    roe = float(row.get('Final ROE') or row.get('ROE') or 0)
    if roe >= 25:   s = 5
    elif roe >= 20: s = 4
    elif roe >= 15: s = 3
    elif roe >= 10: s = 2
    else:           s = 0
    score += s
    details['ROE Score'] = s
    details['ROE Used']  = round(roe, 1)
 
    # ── APPLY MATURITY DISCOUNT ──────────────────────────────
    score = round(score * (1 - discount), 1)
 
    return min(score, 40), details

In [4]:
# ── CELL 4: HISTORICAL SCORE — FINANCIAL COMPANIES (40 pts) ─
# Financial companies (banks, NBFCs) use different metrics
# ROE           : 20 pts (most important for financials)
# Profit growth : 10 pts
# Revenue growth: 10 pts
# Same maturity discount applies
 
def calculate_financial_score(row, raw_data=None):
    """
    Special historical score for financial companies (40 pts)
    ROE weighted heavily (20 pts) as it is the key metric
    Applies same maturity discount
    """
    score   = 0
    details = {}
 
    # ── DATA MATURITY CHECK ──────────────────────────────────
    cagr_years = float(
        row.get('Revenue CAGR Years') or
        row.get('Profit CAGR Years')  or 0
    )
    if cagr_years < 5:
        discount = 0.50
        maturity = 'Very Short (<5Y)'
    elif cagr_years < 8:
        discount = 0.20
        maturity = 'Short (5-8Y)'
    else:
        discount = 0.00
        maturity = 'Full (8Y+)'
 
    details['Data Years'] = cagr_years
    details['Maturity']   = maturity
    details['Discount']   = discount
 
    # ── ROE (20 pts) — most important for financials ─────────
    roe = float(row.get('Final ROE') or row.get('ROE') or 0)
    if roe >= 20:    s = 20
    elif roe >= 18:  s = 17
    elif roe >= 15:  s = 13
    elif roe >= 12:  s = 8
    elif roe >= 10:  s = 4
    else:            s = 0
    score += s
    details['ROE Score'] = s
    details['ROE']       = round(roe, 1)
 
    # ── PROFIT GROWTH (10 pts) ───────────────────────────────
    profit_growth = float(
        row.get('Profit CAGR 5Y')  or
        row.get('Profit CAGR 10Y') or
        row.get('Profit CAGR Max') or 0
    )
    if profit_growth >= 20:  s = 10
    elif profit_growth >= 15:s = 8
    elif profit_growth >= 10:s = 6
    elif profit_growth >= 5: s = 3
    else:                    s = 0
    score += s
    details['Profit Growth Score'] = s
    details['Profit CAGR Used']    = round(profit_growth, 1)
 
    # ── REVENUE GROWTH (10 pts) ──────────────────────────────
    rev_growth = float(
        row.get('Revenue CAGR 5Y')  or
        row.get('Revenue CAGR 10Y') or
        row.get('Revenue CAGR Max') or 0
    )
    if rev_growth >= 20:  s = 10
    elif rev_growth >= 15:s = 8
    elif rev_growth >= 10:s = 6
    elif rev_growth >= 5: s = 3
    else:                 s = 0
    score += s
    details['Revenue Growth Score'] = s
    details['Revenue CAGR Used']    = round(rev_growth, 1)
 
    # ── APPLY MATURITY DISCOUNT ──────────────────────────────
    score = round(score * (1 - discount), 1)
 
    return min(score, 40), details
 
 
print("Historical scoring functions loaded ✅")

Historical scoring functions loaded ✅


In [5]:
# ── CELL 5: PEER COMPARISON SCORE (40 pts) ──────────────────
# Compare each stock against sector peers using percentile rank
# Metrics: ROE, Revenue CAGR, Profit CAGR, OPM, ROCE,
#          Debt/Equity, Revenue YoY, Profit YoY
 
def calculate_peer_scores(df):
    """
    Compare each stock against sector peers (40 pts)
    Score based on percentile rank within sector
    """
    peer_scores = []
 
    peer_metrics = {
        'Final ROE'        : True,   # higher is better
        'Revenue CAGR Max' : True,
        'Profit CAGR Max'  : True,
        'Avg OPM 5Y'       : True,
        'Latest ROCE'      : True,
        'Debt to Equity'   : False,  # lower is better
        'Revenue YoY Q'    : True,
        'Profit YoY Q'     : True,
    }
 
    for _, row in df.iterrows():
        symbol       = row['Symbol']
        sector       = row.get('Sector', 'Unknown')
        scores       = {}
        sector_peers = df[df['Sector'] == sector]
 
        for metric, higher_is_better in peer_metrics.items():
            stock_val = row.get(metric)
 
            if pd.isna(stock_val):
                scores[metric] = None
                continue
 
            peer_vals = sector_peers[metric].dropna()
 
            if len(peer_vals) < 2:
                # Only one stock in sector — give average score
                scores[metric] = 2
                continue
 
            # Percentile rank within sector
            if higher_is_better:
                percentile = (peer_vals < stock_val).sum() / len(peer_vals) * 100
            else:
                percentile = (peer_vals > stock_val).sum() / len(peer_vals) * 100
 
            if percentile >= 80:   score = 5
            elif percentile >= 60: score = 4
            elif percentile >= 40: score = 3
            elif percentile >= 20: score = 2
            else:                  score = 1
 
            scores[metric] = score
 
        valid = [s for s in scores.values() if s is not None]
        if valid:
            raw_total    = sum(valid)
            max_possible = len(valid) * 5
            total        = round(raw_total / max_possible * 40, 1)
        else:
            total        = 20  # neutral if no peer data
 
        peer_scores.append({
            'Symbol'             : symbol,
            'Sector'             : sector,
            'Peer Score'         : total,
            'Peer Count'         : len(sector_peers),
            'ROE Peer Score'     : scores.get('Final ROE'),
            'Revenue Peer Score' : scores.get('Revenue CAGR Max'),
            'Profit Peer Score'  : scores.get('Profit CAGR Max'),
            'Margin Peer Score'  : scores.get('Avg OPM 5Y'),
            'ROCE Peer Score'    : scores.get('Latest ROCE'),
            'Debt Peer Score'    : scores.get('Debt to Equity'),
            'RevYoY Peer Score'  : scores.get('Revenue YoY Q'),
            'ProfYoY Peer Score' : scores.get('Profit YoY Q'),
        })
 
    return pd.DataFrame(peer_scores).sort_values(
        'Peer Score', ascending=False
    ).reset_index(drop=True)
 
 
print("Peer scoring function loaded ✅")

Peer scoring function loaded ✅


In [6]:
# ── CELL 6: BUSINESS QUALITY SCORE (20 pts) ─────────────────
# Promoter holding  : 4 pts (level)
# Promoter trend    : 4 pts (buying/selling)
# Institutional int : 4 pts (low = undiscovered gem)
# Profit consistency: 4 pts
# CF quality        : 4 pts
 
def calculate_quality_score(row):
    """
    Business quality score (20 pts)
    Promoter holding + trend + institutional + consistency + CF
    """
    scores = {}
 
    # ── PROMOTER HOLDING (4 pts) ─────────────────────────────
    promoter = row.get('Promoter Holding')
    level    = score_metric_level(promoter, [60, 50, 40])
    scores['Promoter Holding'] = level if level is not None else 0
 
    # ── PROMOTER TREND (4 pts) ───────────────────────────────
    promoter_change = float(row.get('Promoter Change 4Q') or 0)
    if pd.isna(promoter_change):
        promoter_change = 0
 
    if promoter_change > 1:      pt_score = 4  # Buying ✅
    elif promoter_change > 0:    pt_score = 3  # Slightly buying
    elif promoter_change >= -1:  pt_score = 2  # Stable
    elif promoter_change >= -3:  pt_score = 1  # Slight selling ⚠️
    else:                        pt_score = 0  # Heavy selling ❌
    scores['Promoter Trend'] = pt_score
 
    # ── INSTITUTIONAL INTEREST (4 pts) ───────────────────────
    # Low FII/DII = undiscovered = higher multibagger potential
    # High FII/DII = fully discovered = limited upside
    fii = float(row.get('FII Holding') or 0)
    dii = float(row.get('DII Holding') or 0)
    if pd.isna(fii): fii = 0
    if pd.isna(dii): dii = 0
    institutional = fii + dii
 
    if institutional < 10:   inst_score = 4  # Hidden gem ✅
    elif institutional < 20: inst_score = 3  # Partially discovered
    elif institutional < 35: inst_score = 2  # Well known
    else:                    inst_score = 1  # Fully discovered
    scores['Institutional'] = inst_score
 
    # ── PROFIT CONSISTENCY (4 pts) ───────────────────────────
    profit_q    = row.get('Profit Positive Q', 0)
    total_q     = row.get('Total Quarters',    1)
    consistency = (profit_q / total_q * 100) if total_q > 0 else 0
 
    if consistency >= 95:    pc_score = 4  # Perfect ✅
    elif consistency >= 85:  pc_score = 3
    elif consistency >= 75:  pc_score = 2
    else:                    pc_score = 1
    scores['Profit Consistency'] = pc_score
 
    # ── CASH FLOW QUALITY (4 pts) ────────────────────────────
    cf_years   = row.get('CF Positive Years', 0)
    cf_total   = row.get('CF Total Years',    1)
    cf_ratio   = (cf_years / cf_total * 100) if cf_total > 0 else 0
    cf_growing = row.get('CF Growing')
 
    if cf_ratio >= 90 and cf_growing == True:  cq_score = 4
    elif cf_ratio >= 80:                        cq_score = 3
    elif cf_ratio >= 65:                        cq_score = 2
    else:                                       cq_score = 1
    scores['CF Quality'] = cq_score
 
    total = sum(scores.values())
 
    return {
        'Symbol'                  : row['Symbol'],
        'Sector'                  : row.get('Sector'),
        'Quality Score'           : total,
        'Promoter Holding Score'  : scores['Promoter Holding'],
        'Promoter Trend Score'    : scores['Promoter Trend'],
        'Institutional Score'     : scores['Institutional'],
        'Profit Consistency Score': scores['Profit Consistency'],
        'CF Quality Score'        : scores['CF Quality'],
        'Promoter Holding %'      : promoter,
        'Promoter Change 4Q'      : promoter_change,
        'FII + DII %'             : round(institutional, 2),
    }
 
 
print("Quality scoring function loaded ✅")
 

Quality scoring function loaded ✅


In [7]:
# ── CELL 7: RUN ALL THREE SCORES ────────────────────────────
financial_sectors = ['Financial Services', 'Banking', 'Insurance']
 
# Historical scores
historical_scores = []
for _, row in passed_df.iterrows():
    symbol    = row['Symbol']
    sector    = str(row.get('Sector', ''))
    is_fin    = any(kw in sector for kw in financial_sectors)
 
    if is_fin:
        hist_score, hist_detail = calculate_financial_score(row)
    else:
        hist_score, hist_detail = calculate_historical_score(row)
 
    historical_scores.append({
        'Symbol'          : symbol,
        'Sector'          : sector,
        'Historical Score': hist_score,
        **hist_detail
    })
    print(f"{'FIN' if is_fin else 'STD':3} ✓ {symbol:15} "
          f"Historical Score: {hist_score}/40")
 
historical_df = pd.DataFrame(historical_scores).sort_values(
    'Historical Score', ascending=False
).reset_index(drop=True)
 
# Peer scores
peer_df = calculate_peer_scores(passed_df)
 
# Quality scores
quality_scores = []
for _, row in passed_df.iterrows():
    score = calculate_quality_score(row)
    quality_scores.append(score)
quality_df = pd.DataFrame(quality_scores)
 
print(f"\nAll three scores calculated ✅")
print(f"\n=== TOP 10 HISTORICAL SCORES ===")
for i, row in historical_df.head(10).iterrows():
    print(f"  {i+1:2}. {row['Symbol']:15} {row['Historical Score']}/40"
          f"  {row['Sector']}")

STD ✓ TCS             Historical Score: 24.0/40
STD ✓ INFY            Historical Score: 24.0/40
FIN ✓ HDFCBANK        Historical Score: 28.0/40
STD ✓ RELIANCE        Historical Score: 26.0/40
STD ✓ WIPRO           Historical Score: 17.0/40
STD ✓ PERSISTENT      Historical Score: 37.0/40
STD ✓ LTTS            Historical Score: 21.0/40
STD ✓ COFORGE         Historical Score: 31.0/40
STD ✓ HAPPSTMNDS      Historical Score: 28.8/40
FIN ✓ CHOLAFIN        Historical Score: 40.0/40
FIN ✓ MUTHOOTFIN      Historical Score: 34.0/40
FIN ✓ MANAPPURAM      Historical Score: 10.0/40
FIN ✓ CREDITACC       Historical Score: 10.4/40
STD ✓ GARFIBRES       Historical Score: 24.0/40
STD ✓ SUPRAJIT        Historical Score: 16.0/40
STD ✓ HIMATSEIDE      Historical Score: 18.0/40
STD ✓ HAL             Historical Score: 29.0/40
STD ✓ BEL             Historical Score: 33.0/40
STD ✓ MIDHANI         Historical Score: 8.0/40
STD ✓ PARAS           Historical Score: 27.2/40
STD ✓ VINATIORGA      Historical Score: 1

In [8]:
# ── CELL 8: COMBINE → FINAL SCORE (100 pts) ─────────────────
# Final Score = Historical (40) + Peer (40) + Quality (20)
 
final_df = historical_df[['Symbol', 'Sector', 'Historical Score']].merge(
    peer_df[['Symbol', 'Peer Score']], on='Symbol'
).merge(
    quality_df[['Symbol', 'Quality Score',
                'Promoter Holding %', 'FII + DII %']], on='Symbol'
)
 
final_df['Final Score'] = (
    final_df['Historical Score'] +
    final_df['Peer Score']       +
    final_df['Quality Score']
).round(1)
 
final_df = final_df.sort_values(
    'Final Score', ascending=False
).reset_index(drop=True)
 
print("=== FINAL FUNDAMENTAL RANKING ===")
print(f"{'Rank':<5} {'Symbol':<15} {'Final':>7} "
      f"{'Hist':>6} {'Peer':>6} {'Qual':>6}  {'Sector'}")
print("─" * 75)
for i, row in final_df.iterrows():
    print(f"{i+1:<5} {row['Symbol']:<15}"
          f" {row['Final Score']:>6}/100"
          f" {row['Historical Score']:>5}/40"
          f" {row['Peer Score']:>5}/40"
          f" {row['Quality Score']:>5}/20"
          f"  {row['Sector']}")

=== FINAL FUNDAMENTAL RANKING ===
Rank  Symbol            Final   Hist   Peer   Qual  Sector
───────────────────────────────────────────────────────────────────────────
1     PERSISTENT        76.0/100  37.0/40  28.0/40    11/20  Information Technology
2     CHOLAFIN          75.3/100  40.0/40  26.3/40     9/20  Financial Services
3     MUTHOOTFIN        74.6/100  34.0/40  28.6/40    12/20  Financial Services
4     FINEORG           73.0/100  31.0/40  26.0/40    16/20  Chemicals
5     TITAN             69.0/100  34.0/40  24.0/40    11/20  Consumer Durables
6     DHANUKA           69.0/100  28.0/40  26.0/40    15/20  Chemicals
7     PIIND             68.0/100  33.0/40  23.0/40    12/20  Chemicals
8     BEL               68.0/100  33.0/40  24.0/40    11/20  Capital Goods
9     ABSLAMC           67.0/100  26.0/40  26.0/40    15/20  Financial Services
10    SUNPHARMA         65.0/100  27.0/40  25.0/40    13/20  Healthcare
11    COFORGE           64.0/100  31.0/40  22.0/40    11/20  Informa

In [9]:
# ── CELL 9: SAVE ALL FILES ───────────────────────────────────
historical_df.to_csv('../data/historical_scores.csv', index=False)
peer_df.to_csv('../data/peer_scores.csv',             index=False)
quality_df.to_csv('../data/quality_scores.csv',       index=False)
final_df.to_csv('../data/fundamental_scores.csv',     index=False)
 
print(f"All score files saved ✅")
print(f"\nTop 10 Fundamental Picks:")
for i, row in final_df.head(10).iterrows():
    print(f"  {i+1:2}. {row['Symbol']:15} "
          f"{row['Final Score']}/100  {row['Sector']}")
 

All score files saved ✅

Top 10 Fundamental Picks:
   1. PERSISTENT      76.0/100  Information Technology
   2. CHOLAFIN        75.3/100  Financial Services
   3. MUTHOOTFIN      74.6/100  Financial Services
   4. FINEORG         73.0/100  Chemicals
   5. TITAN           69.0/100  Consumer Durables
   6. DHANUKA         69.0/100  Chemicals
   7. PIIND           68.0/100  Chemicals
   8. BEL             68.0/100  Capital Goods
   9. ABSLAMC         67.0/100  Financial Services
  10. SUNPHARMA       65.0/100  Healthcare
